### Using web scraping tools, extract the content from a set of web pages (e.g., news, blogs). Analyse the text to identify trends and topics.

In [1]:
import requests
import re
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF

# 1. Web pages to analyse
urls = [
    "https://www.bbc.com/news/technology",
    "https://www.cnn.com/tech"
]

# 2. Scrape text from web pages
def scrape_webpage(url):
    try:
        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=15
        )
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        paragraphs = soup.find_all("p")
        text = " ".join(p.get_text(" ", strip=True) for p in paragraphs)
        return text

    except requests.RequestException as e:
        print(f"Could not access {url}: {e}")
        return ""

web_texts = [scrape_webpage(url) for url in urls]

# Remove pages where scraping failed
documents = [text for text in web_texts if text.strip()]

if len(documents) < 2:
    raise ValueError(
        "Could not scrape enough pages. Try other accessible article URLs."
    )

# 3. Preprocess text
def preprocess(text):
    text = text.lower()
    words = re.findall(r"\b[a-z]+\b", text)
    words = [
        word for word in words
        if word not in ENGLISH_STOP_WORDS and len(word) > 2
    ]
    return " ".join(words)

processed_texts = [preprocess(text) for text in documents]

# 4. Convert text into TF-IDF features
vectorizer = TfidfVectorizer(max_df=1.0, min_df=1)
tfidf = vectorizer.fit_transform(processed_texts)
feature_names = vectorizer.get_feature_names_out()

# 5. Perform topic modelling using NMF
num_topics = min(3, tfidf.shape[0], tfidf.shape[1])
nmf_model = NMF(n_components=num_topics, random_state=42, max_iter=500)
nmf_model.fit(tfidf)

# 6. Display top words in each topic
print("\n--- Topics Identified ---")
for topic_idx, topic in enumerate(nmf_model.components_):
    top_indices = topic.argsort()[-10:][::-1]
    top_words = [feature_names[i] for i in top_indices]
    print(f"\nTopic {topic_idx + 1}:")
    print(", ".join(top_words))


--- Topics Identified ---

Topic 1:
jones, dow, llc, indices, cnn, provided, market, reserved, news, rights

Topic 2:
bbc, tech, government, solar, said, taken, man, meta, using, external
